<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### My Lane: Ranking Signal Analysis

**Question:** Which signals are associated with page performance?

**Task Type:** Signal analysis (primary) + Ranking (secondary)

### Why Three Models?

I use three models to get a complete picture:

| Model | What It Tells Me | Why It Fits My Lane |
|---|---|---|
| **Logistic Regression** | Coefficients (direction + magnitude of each signal) | Answers "which signals push the needle?" |
| **Decision Tree (depth 3)** | Interactions between signals (readable flowchart) | Answers "how do signals work together?" |
| **Random Forest** | Feature importance (which signals matter most) | Answers "what should we focus on?" |

### Why These Models Fit My Question

My lane asks: "Which signals are associated with page performance?"

- **Logistic Regression** → Shows if a signal increases or decreases decline risk
- **Decision Tree** → Shows how signals combine (e.g., age + position together)
- **Random Forest** → Shows which signals are most important overall

### What I'll Compare

All models will be compared against my Week 4 baseline on the same metric: **Precision@20** and **Precision@50**.

The baseline rule is: `score = stale × visible × impressions` (captures 17 pages).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Validation Strategy: Client-Holdout (80/20 split by client_id)

**Why client-holdout?**
- Prevents the model from memorizing client-specific patterns
- Tests whether the model can generalize to unseen clients
- Matches real-world deployment (new clients will be unseen)

**Why 80/20?**
- 30,000 rows → 24,000 train, 6,000 test
- Enough data to train well
- Enough test data to evaluate reliably
- Matches the starter pipeline's approach

---

### Data Preparation

**Features (all knowable at decision time):**

| Feature | Why It's a Feature |
|---|---|
| `avg_position` | Average search position — knowable from feature window |
| `ctr` | Click-through rate — knowable from feature window |
| `engagement_rate` | Visitor engagement — knowable from feature window |
| `content_age_days` | Age of content — knowable at decision point |
| `word_count` | Content length — knowable at decision point |
| `search_volume` | Keyword demand — knowable from feature window |
| `competition` | Competition level — knowable from feature window |
| `days_since_last_update` | Days since update — knowable at decision point |
| `content_type` | Type of page (categorical) — knowable at decision point |
| `main_intent` | User intent (categorical) — knowable at decision point |

**Label:**
- `is_declining_label` — whether the page is declining (observed outcome)

**Excluded (Leakage Prevention):**

| Column | Why Excluded |
|---|---|
| `trend_pct` | Derived from the label — LEAKAGE! |
| `trend_direction` | Derived from the label — LEAKAGE! |
| `content_id` | Identifier, NOT a signal |
| `client_id` | Used ONLY for splitting, NOT as a feature |

**Missing Value Handling:**
- `avg_position`: 0 means "no data" → replaced with -1
- Other numeric columns: filled with median
- Categorical columns: one-hot encoded

---

### Implementation

- 80% of clients → Training set
- 20% of clients → Test set
- Client_id used ONLY for splitting, NOT as a feature

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Loaded {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}\n")

# print(df["impressions_90d"])
# print(df["days_since_last_update"])
# stale = (df["days_since_last_update"] >= 180).astype(int)
# visible = (df["impressions_90d"] >= 500).astype(int)
# print(stale)
# print(visible)

# ============================================================
# 1. Baseline (from Week 4)
# ============================================================

print("=" * 60)
print("Baseline (Week 4 Rule)")
print("=" * 60)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()

base_prec20 = precision_at_k(df["baseline_score"], df["is_declining_label"], 20)
base_prec50 = precision_at_k(df["baseline_score"], df["is_declining_label"], 50)

print(f"Baseline Precision@20: {base_prec20:.3f}")
print(f"Baseline Precision@50: {base_prec50:.3f}")
print(f"Pages with score > 0: {(df['baseline_score'] > 0).sum()}")

# df.info()


# ============================================================
# 2. Prepare Data for Modeling
# ============================================================

print("\n" + "=" * 60)
print("Prepare Data for Modeling")
print("=" * 60)

# Features (ALL knowable at decision time)
# Updated to include more safe signals
numeric_features = [
    'avg_position',           # Search rank
    'ctr',                    # Click-through rate
    'engagement_rate',        # Visitor engagement
    'content_age_days',       # Age of content
    'word_count',             # Content length
    'search_volume',          # Keyword demand
    'competition',            # Competition level
    'days_since_last_update', # Days since update
    'scroll_rate',            # How far people scroll (content quality)
    'days_with_impressions'   # Consistency of visibility
]

categorical_features = [
    'content_type',           # Type of page
    'main_intent',            # User intent
    'impression_tier',        # Impression bucket
    'position_tier'           # Position bucket
]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features: {len(numeric_features) + len(categorical_features)}")

# HANDLE MISSING VALUES IN NUMERIC COLUMNS (before preprocessing)
# We need to clean the numeric columns in the original df
df['avg_position'] = df['avg_position'].replace(0, -1)  # 0 = no data

for col in numeric_features:
    if col != 'avg_position':
        df[col] = df[col].fillna(df[col].median())

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),  # Scales numeric features
#Converts categories to 0/1
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

# Pass the FULL dataframe (with ALL columns)
X = preprocessor.fit_transform(df)
y = df['is_declining_label'].values

# # Check what's in X
# print(f"X shape: {X.shape}")
# print(f"Number of features in X: {X.shape[1]}")

# # Get feature names
# feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
# print(f"Total features: {len(feature_names)}")
# print(f"Feature names (first 5): {feature_names[:5]}")

# # Show first row of X
# print(f"\nFirst row of X (first 10 values): {X[0, :10]}")

# Split by client (client-holdout)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(train_idx):,} rows, Test: {len(test_idx):,} rows")
print(f"Test declining rate: {y_test.mean():.3f}")


# ============================================================
# 3. Train All 3 Models
# ============================================================

print("\n" + "=" * 60)
print("Train All 3 Models")
print("=" * 60)

# Model 1: Logistic Regression
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train, y_train) # Train the algorithm on the training data

# Model 2: Decision Tree (depth 3 - readable!)
dt = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train) # Train the algorithm on the training data

# Model 3: Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train) # Train the algorithm on the training data

print("All 3 models trained successfully!")


# ============================================================
# 4. Compare vs Baseline
# ============================================================

print("\n" + "=" * 60)
print("Compare vs Baseline")
print("=" * 60)

def precision_at_k_proba(model, X, y, k):
    scores = model.predict_proba(X)[:, 1]
    return precision_at_k(scores, y, k)

models = {
    'Baseline (my rule)': (df['baseline_score'].values, None),
    # Make prediction with the trained model using test data
    'Logistic Regression': (lr, X_test),
    'Decision Tree': (dt, X_test),
    'Random Forest': (rf, X_test)
}

print("\nPrecision Comparison Table:")
print("-" * 70)
print(f"{'Method':<22} {'Precision@20':<14} {'Precision@50':<14}")
print("-" * 70)

results = {}

for name, (model_or_scores, X_data) in models.items():
    if name == 'Baseline (my rule)':
        prec20 = precision_at_k(model_or_scores, y, 20)
        prec50 = precision_at_k(model_or_scores, y, 50)
    else:
        prec20 = precision_at_k_proba(model_or_scores, X_data, y_test, 20)
        prec50 = precision_at_k_proba(model_or_scores, X_data, y_test, 50)
    results[name] = {'Precision@20': prec20, 'Precision@50': prec50}
    print(f"{name:<22} {prec20:<14.3f} {prec50:<14.3f}")

print("-" * 70)
print(f"\nBase Rate (overall declining rate): {y.mean():.3f}")


# ============================================================
# 5. Print the Decision Tree (Human-Readable!)
# ============================================================

print("\n" + "=" * 60)
print("Decision Tree (Depth 3) — Human-Readable")
print("=" * 60)

feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
print(export_text(dt, feature_names=feature_names))

# ============================================================
# 6. Feature Importance (Random Forest)
# ============================================================

print("\n" + "=" * 60)
print("Random Forest — Feature Importance")
print("=" * 60)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nTop 5 Most Important Features:")
for i in range(min(5, len(feature_names))):
    idx = indices[i]
    print(f"  {i+1}. {feature_names[idx]}: {importances[idx]:.3f}")

# ============================================================
# 7. Logistic Regression Coefficients
# ============================================================

print("\n" + "=" * 60)
print("Logistic Regression — Coefficients")
print("=" * 60)

coefs = lr.coef_[0]
top_coef_indices = np.argsort(np.abs(coefs))[::-1]

print("\nTop 5 Features by Coefficient Magnitude:")
for i in range(min(5, len(feature_names))):
    idx = top_coef_indices[i]
    direction = "INCREASES decline" if coefs[idx] > 0 else "DECREASES decline"
    print(f"  {i+1}. {feature_names[idx]}: {coefs[idx]:.3f} ({direction})")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded 30,000 rows
Declining rate: 0.542

Baseline (Week 4 Rule)
Baseline Precision@20: 0.900
Baseline Precision@50: 0.680
Pages with score > 0: 17

Prepare Data for Modeling
Numeric features: 10
Categorical features: 4
Total features: 14
Train: 23,837 rows, Test: 6,163 rows
Test declining rate: 0.511

Train All 3 Models
All 3 models trained successfully!

Compare vs Baseline

Precision Comparison Table:
----------------------------------------------------------------------
Method                 Precision@20   Precision@5

# Baseline vs Model Comparison

## Results

### Comparison Table

| Method | Precision@20 | Precision@50 |
|---|---|---|
| Baseline (my rule) | **0.900** | 0.680 |
| Logistic Regression | 0.650 | **0.700** |
| Decision Tree | 0.550 | 0.480 |
| Random Forest | 0.450 | 0.620 |

*Base Rate (overall declining rate): 0.542*

---

## Key Takeaways

### 1. Baseline Dominates at the Top
The simple rule (`stale × visible × impressions`) achieves **90% precision at top 20** — meaning 18 of the top 20 recommendations are actually declining pages. This shows that the combination of **staleness (180+ days)** and **visibility (500+ impressions)** is an exceptionally strong signal for finding the most critical pages.

### 2. Logistic Regression Wins at Scale
At top 50, Logistic Regression achieves **70% precision**, slightly beating the baseline (68%). This means it identifies **35 correct pages** vs the baseline's 34 — a small but real improvement. The expanded feature set (14 features including `scroll_rate`, `days_with_impressions`, and position/impression tiers) helped the model find **additional declining pages** beyond what the simple rule captures.

### 3. Decision Tree and Random Forest Underperformed
Both tree-based models performed worse than the baseline, possibly due to:
- **Overfitting** to training data
- **The depth constraint** (Decision Tree limited to depth 3)
- **Noise in additional features** confusing the models

---

## What the Models Learned

| Signal | Importance | Direction |
|---|---|---|
| **Position** | Most important | Top 3 positions → **decreases** decline risk |
| **Consistency** | 2nd most important | Consistent visibility → mixed effect (can indicate slow decline) |
| **Content Age** | 3rd most important | Complex effect — depends on other factors |

**Feature Importance (Random Forest):**
1. `avg_position` — strongest predictor
2. `days_with_impressions` — consistency matters
3. `content_age_days` — age matters
4. `word_count` — length has some signal
5. `ctr` — CTR matters but less than position

**Logistic Regression Coefficients:**
- `position_tier_top_3`: **-1.019** (DECREASES decline — top 3 is much safer)
- `impression_tier_excellent`: **-0.770** (DECREASES decline)
- `days_with_impressions`: **+0.636** (INCREASES decline — consistent visibility can signal decline)
- `content_age_days`: **-0.431** (DECREASES decline — older pages less likely to decline when controlling for other factors)

---

## Conclusion

**Use both:**
- **Baseline rule** for the **top 20** (highest precision)
- **Logistic Regression** for the **next 30** (better coverage)

The baseline rule remains a strong, simple option — especially when only the most critical pages are needed. However, Logistic Regression with expanded features can identify additional declining pages when more candidates are required.

> *"My baseline rule performed exceptionally well at the top 20 (90% precision), beating all models. However, Logistic Regression slightly improved performance at the top 50 (70% vs 68%), suggesting it can identify additional declining pages that the simple rule misses. This shows that while the rule is excellent for finding the most critical pages, a learned model can expand coverage."*

---

## Model Interpretation

### Decision Tree (Readable Rule)
The tree shows only **one path** that predicts decline:

> `days_with_impressions > -1.13` AND `content_age_days <= 0.67`

This means pages that are **consistently visible** but **relatively young** are most likely to be declining.

### Feature Importance (Random Forest)
Position (`avg_position`) is the **strongest predictor**, followed by consistency (`days_with_impressions`) and content age (`content_age_days`). This aligns with the baseline's focus on visibility and staleness.

### Coefficients (Logistic Regression)
- **Top 3 position** is the most protective factor (-1.019 coefficient)
- **Excellent impressions** also protect against decline (-0.770)
- **Consistent visibility** can signal decline (+0.636)
- **Content age** shows mixed effects depending on other factors

---

## What This Means for Content Teams

| Finding | Action |
|---|---|
| **Position matters most** | Pages dropping position need immediate attention |
| **Stale + visible = high risk** | Old pages with traffic are prime refresh candidates |
| **Consistency can signal decline** | Pages seen regularly but declining need investigation |
| **Top 3 pages are safe** | Focus efforts on pages outside top 3 |

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [16]:
# ============================================================
# Errors and Interpretation
# ============================================================

print("\n" + "=" * 60)
print("Errors and Interpretation")
print("=" * 60)

# Get predictions from Logistic Regression (best model)
lr_probs = lr.predict_proba(X_test)[:, 1]
lr_preds = lr.predict(X_test)

# Find where the model made errors
errors = X_test[(lr_preds != y_test)]
error_indices = np.where(lr_preds != y_test)[0]

print(f"\nTotal errors on test set: {len(errors)} out of {len(X_test)} rows")
print(f"Error rate: {len(errors)/len(X_test):.3f}")

# ============================================================
# Show 3 Concrete Wrong Cases
# ============================================================

print("\n" + "=" * 60)
print("3 Concrete Wrong Cases (Logistic Regression)")
print("=" * 60)

# Get the test data indices
test_df = df.iloc[test_idx].copy()
test_df['lr_pred'] = lr_preds
test_df['lr_prob'] = lr_probs
test_df['actual'] = y_test

# Find false positives (predicted decline, actually stable)
false_positives = test_df[(test_df['lr_pred'] == 1) & (test_df['actual'] == 0)]

# Find false negatives (predicted stable, actually declining)
false_negatives = test_df[(test_df['lr_pred'] == 0) & (test_df['actual'] == 1)]

print(f"\nFalse Positives (model said 'declining' but page was stable): {len(false_positives)}")
print(f"False Negatives (model said 'stable' but page was declining): {len(false_negatives)}")

# Show 3 examples of each (if available)
print("\n--- False Positives (Top 3) ---")
if len(false_positives) > 0:
    for idx, (i, row) in enumerate(false_positives.head(3).iterrows()):
        print(f"\n{idx+1}. Content ID: {row['content_id']}")
        print(f"   Actual: Stable (0), Predicted: Declining (1)")
        print(f"   Probability: {row['lr_prob']:.3f}")
        print(f"   Key signals:")
        print(f"     - avg_position: {row['avg_position']:.1f}")
        print(f"     - ctr: {row['ctr']:.3f}")
        print(f"     - content_age_days: {row['content_age_days']:.0f}")
        print(f"     - days_with_impressions: {row['days_with_impressions']:.0f}")
else:
    print("   No false positives found!")

print("\n--- False Negatives (Top 3) ---")
if len(false_negatives) > 0:
    for idx, (i, row) in enumerate(false_negatives.head(3).iterrows()):
        print(f"\n{idx+1}. Content ID: {row['content_id']}")
        print(f"   Actual: Declining (1), Predicted: Stable (0)")
        print(f"   Probability: {row['lr_prob']:.3f}")
        print(f"   Key signals:")
        print(f"     - avg_position: {row['avg_position']:.1f}")
        print(f"     - ctr: {row['ctr']:.3f}")
        print(f"     - content_age_days: {row['content_age_days']:.0f}")
        print(f"     - days_with_impressions: {row['days_with_impressions']:.0f}")
else:
    print("   No false negatives found!")

print("\n" + "=" * 60)
print("Summary of Errors")
print("=" * 60)

print(f"\nTotal errors: {len(errors)}")
print(f"False Positives: {len(false_positives)} (predicted decline, actually stable)")
print(f"False Negatives: {len(false_negatives)} (predicted stable, actually declining)")
print(f"\nThe model struggles with edge cases where signals are contradictory.")
print("This suggests additional signals (competition trends, seasonality) could improve performance.")


Errors and Interpretation

Total errors on test set: 2745 out of 6163 rows
Error rate: 0.445

3 Concrete Wrong Cases (Logistic Regression)

False Positives (model said 'declining' but page was stable): 1092
False Negatives (model said 'stable' but page was declining): 1653

--- False Positives (Top 3) ---

1. Content ID: content_a5a2fbc76336
   Actual: Stable (0), Predicted: Declining (1)
   Probability: 0.596
   Key signals:
     - avg_position: 39.8
     - ctr: 0.000
     - content_age_days: 238
     - days_with_impressions: 69

2. Content ID: content_72c5c2d73e5a
   Actual: Stable (0), Predicted: Declining (1)
   Probability: 0.561
   Key signals:
     - avg_position: 30.0
     - ctr: 0.120
     - content_age_days: 300
     - days_with_impressions: 88

3. Content ID: content_bce275871a25
   Actual: Stable (0), Predicted: Declining (1)
   Probability: 0.632
   Key signals:
     - avg_position: 5.4
     - ctr: 1.350
     - content_age_days: 187
     - days_with_impressions: 82

--- F

## Errors and Interpretation

### Where is the Model Wrong?

The Logistic Regression model makes errors on **44.5%** of test cases, with more false negatives (1,653) than false positives (1,092). This means the model is more likely to **miss declining pages** than to falsely flag stable ones.

### Common Error Patterns

| Error Type | Pattern | Example |
|---|---|---|
| **False Positive** | Very poor signals but stable | Page with position 39.8, 0% CTR, but stable — likely a niche topic with consistent demand |
| **False Positive** | Good signals but flagged due to age | Page with position 5.4, CTR 1.35%, but old (187 days) — model over-indexed on age |
| **False Negative** | Poor signals but predicted stable | Page with position 20.3, 0.05% CTR, very old (445 days) — model over-indexed on consistency |
| **False Negative** | Mixed signals | Page with good position (7.2) but 0% CTR and inconsistent visibility — model over-indexed on position |

### Key Patterns in Errors

| Pattern | Why It Happens |
|---|---|
| **Over-indexing on age** | Old pages get flagged as declining even with good signals |
| **Over-indexing on consistency** | Consistent visibility masks decline signals |
| **Under-indexing on CTR** | Zero CTR with good position is a decline risk the model misses |
| **Mixed signals confuse the model** | Pages with good position but bad CTR are ambiguous |

### What the Model Learns (and Misses)

**The model learns:**
- Poor position + old age = decline risk
- Consistency is a strong signal

**The model misses:**
- Pages that are "always bad" (stable low-performers)
- Declining pages with good position but bad CTR
- Declining pages with consistent visibility

### What This Means in Practice

| Finding | Action |
|---|---|
| **Poor position + old age** | High confidence decline — review immediately |
| **Good position + bad CTR** | Monitor closely — potential decline pattern |
| **Consistent visibility** | Don't assume stability — investigate further |
| **Mixed signals** | Manual review needed — model can't resolve ambiguity |

### One Sentence Summary

> The model is strongest at identifying clear decline patterns (poor position + old age) but struggles with edge cases where signals are contradictory, particularly confusing "always bad" pages with "getting worse" pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.